In [ ]:
import cvxpy as cp
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def solve_diet_problem(csv_filepath, is_vegan=False):
    """
    Solves the MILP Diet Problem.
    Args:
        csv_filepath (str): Path to price data.
        is_vegan (bool): If True, disables Dairy requirement and Logic constraints.
    Returns:
        dict: Solution details (status, total_cost, basket_df).
    """
    # 1. Load Data
    try:
        df = pd.read_csv(csv_filepath)
    except FileNotFoundError:
        return {"status": "File Not Found"}

    # --- DYNAMIC COLUMN MAPPING ---
    # Find the column that contains 'Price', 'Protein', 'Calorie'
    try:
        price_col = [c for c in df.columns if 'Price' in c][0]
        prot_col = [c for c in df.columns if 'Protein' in c][0]
        cal_col = [c for c in df.columns if 'Calorie' in c][0]
    except IndexError:
        return {"status": "Error: Columns not found. Check CSV headers."}

    prices = df[price_col].values
    protein = df[prot_col].values
    calories = df[cal_col].values
    names = df['Item'].values

    # Helper: Check category (Case insensitive)
    def is_cat(i, tag):
        # Handle NaN categories gracefully
        cat_val = str(df.loc[i, 'Category'])
        return tag.lower() in cat_val.lower()

    n = len(df)
    M = 50  # Big-M: Max physical units per item

    # 2. Variables
    x = cp.Variable(n, integer=True) # Quantity
    y = cp.Variable(n, boolean=True)              # Binary Indicator

    # 3. Constraints
    constraints = []
    constraints.append(x >= y) # If selected (y=1), must buy at least 1 unit (x>=1)
    constraints.append(x >= 0) # Non-negativity

    # A. Nutritional Bounds (WHO Average)
    constraints.append(x @ protein >= 378)     # 54g/day * 7
    constraints.append(x @ calories >= 15750)  # 2250kcal/day * 7

    # B. Backpack Constraint
    constraints.append(cp.sum(x) <= 15)

    # C. Big-M Linking
    constraints.append(x <= M * y)

    # D. Category Diversity
    # Vegetables (>= 2)
    idx_veg = [i for i in range(n) if is_cat(i, 'Vegetables')]
    if idx_veg: constraints.append(cp.sum(y[idx_veg]) >= 2)

    # Fruits (>= 1)
    idx_fruit = [i for i in range(n) if is_cat(i, 'Fruits')]
    if idx_fruit: constraints.append(cp.sum(y[idx_fruit]) >= 1)

    # Protein (>= 2)
    idx_prot = [i for i in range(n) if is_cat(i, 'Protein')]
    if idx_prot: constraints.append(cp.sum(y[idx_prot]) >= 2)

    # Dairy (Conditional)
    idx_dairy = [i for i in range(n) if is_cat(i, 'Dairy')]
    if not is_vegan and idx_dairy:
        constraints.append(cp.sum(y[idx_dairy]) >= 1)
    elif is_vegan and idx_dairy:
        constraints.append(cp.sum(x[idx_dairy]) == 0) # Force 0 dairy

    # E. Logical Dependencies (Strict Necessity)
    def get_idx(substring):
        matches = [i for i, name in enumerate(names) if substring.lower() in name.lower()]
        return matches[0] if matches else None

    # Logic 1 & 2: Oats/Cereal -> Milk (Only if not Vegan)
    # Rationale: Dry cereal/oats require a liquid base (Milk).
    if not is_vegan:
        i_milk = get_idx('Milk')

        # 1. Oats -> Milk
        i_oats = get_idx('Oats')
        if i_oats is not None and i_milk is not None:
             constraints.append(y[i_oats] <= y[i_milk])

        # 2. Cereal -> Milk
        i_cereal = get_idx('Cereal')
        # Also check plural 'Cereals' just in case
        if i_cereal is None: i_cereal = get_idx('Cereals')

        if i_cereal is not None and i_milk is not None:
             constraints.append(y[i_cereal] <= y[i_milk])

    # Logic 3: Pasta -> Oil
    # Rationale: Pasta requires oil (or fat) to prevent sticking/cooking.
    i_pasta = get_idx('Pasta')
    i_oil = get_idx('Oil')
    if i_pasta is not None and i_oil is not None:
        constraints.append(y[i_pasta] <= y[i_oil])

    # 4. Objective
    objective = cp.Minimize(x @ prices)

    # 5. Solve (Using Default Solver)
    prob = cp.Problem(objective, constraints)
    prob.solve()

    # 6. Package Results
    result = {
        "status": prob.status,
        "cost": prob.value if prob.status == 'optimal' else None,
        "basket": None
    }

    if prob.status == 'optimal':
        df['Qty'] = np.round(x.value)
        basket = df[df['Qty'] > 0].copy()
        # Use dynamic price column to avoid KeyError
        basket['Total_Cost'] = basket['Qty'] * basket[price_col]
        result['basket'] = basket

    return result

In [ ]:
# --- Scenario 1: Buenos Aires (Standard) ---
res_ba = solve_diet_problem('prices_ba.csv', is_vegan=False)

print(f"--- BUENOS AIRES RESULTS ---")
print(f"Status: {res_ba['status']}")
if res_ba['cost']:
    print(f"Total Cost: ${res_ba['cost']:.2f}")
    if res_ba['basket'] is not None:
        # Select specific columns to print
        cols = ['Item', 'Qty', 'Total_Cost', 'Category']
        print(res_ba['basket'][cols])

In [ ]:
# --- Scenario 2: San Francisco (Standard) ---
res_sf = solve_diet_problem('prices_sf.csv', is_vegan=False)

print(f"\n--- SAN FRANCISCO RESULTS ---")
print(f"Status: {res_sf['status']}")
if res_sf['cost']:
    print(f"Total Cost: ${res_sf['cost']:.2f}")
    if res_sf['basket'] is not None:
         cols = ['Item', 'Qty', 'Total_Cost', 'Category']
         print(res_sf['basket'][cols])

In [ ]:
# --- Scenario 3: Vegan Sensitivity Test (Using BA Data) ---
res_vegan = solve_diet_problem('prices_ba.csv', is_vegan=True)

print(f"\n--- VEGAN SENSITIVITY TEST (BA) ---")
print(f"Status: {res_vegan['status']}")
if res_vegan['cost']:
    print(f"Total Cost: ${res_vegan['cost']:.2f}")
    if res_vegan['basket'] is not None:
        print(res_vegan['basket'][['Item', 'Qty', 'Category']])

In [ ]:
def plot_results(basket_ba, basket_sf):
    # Aggregate costs by Category
    cat_ba = basket_ba.groupby('Category')['Total_Cost'].sum()
    cat_sf = basket_sf.groupby('Category')['Total_Cost'].sum()

    # Normalize to % of Total Budget for comparison (Optional)
    # cat_ba = cat_ba / cat_ba.sum() * 100
    # cat_sf = cat_sf / cat_sf.sum() * 100

    # Align categories
    all_cats = sorted(list(set(cat_ba.index) | set(cat_sf.index)))
    ba_vals = [cat_ba.get(c, 0) for c in all_cats]
    sf_vals = [cat_sf.get(c, 0) for c in all_cats]

    # Plot
    x = np.arange(len(all_cats))
    width = 0.35

    fig, ax1 = plt.subplots(figsize=(10, 6))

    # Plot BA on primary axis
    ax1.bar(x - width/2, ba_vals, width, label='Buenos Aires (ARS)', color='skyblue')
    ax1.set_ylabel('Cost (ARS)', color='skyblue', fontweight='bold')

    # Create secondary axis for SF (USD) so bars are visible despite currency difference
    ax2 = ax1.twinx()
    ax2.bar(x + width/2, sf_vals, width, label='San Francisco (USD)', color='orange')
    ax2.set_ylabel('Cost (USD)', color='orange', fontweight='bold')

    ax1.set_xticks(x)
    ax1.set_xticklabels([c.split(';')[0] for c in all_cats], rotation=45) # Simplify labels
    ax1.set_title('Cost Allocation by Category: BA vs SF')

    fig.legend(loc='upper right', bbox_to_anchor=(0.9, 0.85))
    plt.tight_layout()
    plt.show()

# Only run if both solved successfully
if res_ba['status'] == 'optimal' and res_sf['status'] == 'optimal':
    plot_results(res_ba['basket'], res_sf['basket'])

In [ ]:
import seaborn as sns

def plot_comparative_pies(basket_ba, basket_sf):
    """Plot side-by-side pie charts for BA vs SF cost allocation."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # Prepare data
    ba_data = basket_ba.groupby('Category')['Total_Cost'].sum()
    sf_data = basket_sf.groupby('Category')['Total_Cost'].sum()

    # Get consistent colors
    categories = sorted(set(ba_data.index) | set(sf_data.index))
    colors = plt.cm.Set3(np.linspace(0, 1, len(categories)))
    color_map = dict(zip(categories, colors))

    # BA pie
    axes[0].pie(ba_data, labels=ba_data.index, autopct='%1.1f%%',
                colors=[color_map[c] for c in ba_data.index])
    axes[0].set_title('Buenos Aires (ARS)', fontweight='bold')

    # SF pie
    axes[1].pie(sf_data, labels=sf_data.index, autopct='%1.1f%%',
                colors=[color_map[c] for c in sf_data.index])
    axes[1].set_title('San Francisco (USD)', fontweight='bold')

    plt.suptitle('Cost Allocation by Category', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

# Execute if optimal
if res_ba['status'] == 'optimal' and res_sf['status'] == 'optimal':
    plot_comparative_pies(res_ba['basket'], res_sf['basket'])